# Task 12 - Unsupervised Learning & Dimensionality Reduction
## GDGOC AI/ML Fellowship — Week 7

**Topics Covered:**
- K-Means Clustering (from Scratch)
- Hierarchical Clustering
- PCA (Principal Component Analysis)
- t-SNE Visualization
- Association Rule Mining (Apriori)
- **Main Project: Market Basket Analysis**

---
## 1. Install & Import Libraries

In [ ]:
!pip install mlxtend pandas numpy matplotlib seaborn scikit-learn scipy

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

import warnings
warnings.filterwarnings('ignore')
print('All libraries imported successfully!')

---
## 2. K-Means Clustering from Scratch

K-Means is an unsupervised algorithm that groups data into **K clusters** by minimizing the distance between points and their cluster centroids.

**Steps:**
1. Initialize K random centroids
2. Assign each point to nearest centroid
3. Recompute centroids
4. Repeat until convergence

In [ ]:
class KMeansScratch:
    def __init__(self, k=3, max_iters=100, tol=1e-4):
        self.k = k
        self.max_iters = max_iters
        self.tol = tol

    def fit(self, X):
        # Randomly initialize centroids
        np.random.seed(42)
        indices = np.random.choice(len(X), self.k, replace=False)
        self.centroids = X[indices]

        for _ in range(self.max_iters):
            # Assign clusters
            self.labels = self._assign_clusters(X)
            # Update centroids
            new_centroids = np.array([X[self.labels == i].mean(axis=0) for i in range(self.k)])
            # Check convergence
            if np.linalg.norm(new_centroids - self.centroids) < self.tol:
                break
            self.centroids = new_centroids
        return self

    def _assign_clusters(self, X):
        distances = np.array([[np.linalg.norm(x - c) for c in self.centroids] for x in X])
        return np.argmin(distances, axis=1)

    def predict(self, X):
        return self._assign_clusters(X)


# Generate sample data
from sklearn.datasets import make_blobs
X_blobs, y_true = make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=42)

# Fit KMeans from scratch
km = KMeansScratch(k=4)
km.fit(X_blobs)
labels = km.labels

# Plot
plt.figure(figsize=(8, 5))
colors = ['red', 'blue', 'green', 'purple']
for i in range(4):
    plt.scatter(X_blobs[labels==i, 0], X_blobs[labels==i, 1], s=40, label=f'Cluster {i+1}')
plt.scatter(km.centroids[:, 0], km.centroids[:, 1], s=200, c='black', marker='X', label='Centroids')
plt.title('K-Means Clustering from Scratch')
plt.legend()
plt.tight_layout()
plt.show()
print('K-Means from scratch complete!')

---
## 3. Hierarchical Clustering

Hierarchical Clustering builds a **tree (dendrogram)** of clusters. Unlike K-Means, you don't need to specify K in advance.

- **Agglomerative**: Bottom-up (each point starts as its own cluster)
- **Divisive**: Top-down

In [ ]:
# Dendrogram
plt.figure(figsize=(10, 5))
linked = linkage(X_blobs[:50], method='ward')
dendrogram(linked, truncate_mode='level', p=4)
plt.title('Hierarchical Clustering Dendrogram')
plt.xlabel('Sample Index')
plt.ylabel('Distance')
plt.tight_layout()
plt.show()

# Agglomerative Clustering
hc = AgglomerativeClustering(n_clusters=4, linkage='ward')
hc_labels = hc.fit_predict(X_blobs)

plt.figure(figsize=(8, 5))
for i in range(4):
    plt.scatter(X_blobs[hc_labels==i, 0], X_blobs[hc_labels==i, 1], s=40, label=f'Cluster {i+1}')
plt.title('Hierarchical Clustering Result')
plt.legend()
plt.tight_layout()
plt.show()

---
## 4. PCA — Principal Component Analysis

PCA reduces dimensionality by finding **principal components** — directions of maximum variance in the data.

**Why use PCA?**
- Remove redundant features
- Speed up training
- Enable 2D/3D visualization of high-dimensional data

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris()
X_iris = iris.data
y_iris = iris.target

# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_iris)

# Apply PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(f'Explained Variance Ratio: {pca.explained_variance_ratio_}')
print(f'Total Variance Explained: {sum(pca.explained_variance_ratio_)*100:.2f}%')

# Plot
plt.figure(figsize=(8, 5))
for i, name in enumerate(iris.target_names):
    plt.scatter(X_pca[y_iris==i, 0], X_pca[y_iris==i, 1], label=name, s=60)
plt.title('PCA — Iris Dataset (2 Components)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend()
plt.tight_layout()
plt.show()

---
## 5. t-SNE Visualization

t-SNE (t-Distributed Stochastic Neighbor Embedding) is a **non-linear** dimensionality reduction technique great for visualizing clusters in high-dimensional data.

In [ ]:
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_tsne = tsne.fit_transform(X_scaled)

plt.figure(figsize=(8, 5))
for i, name in enumerate(iris.target_names):
    plt.scatter(X_tsne[y_iris==i, 0], X_tsne[y_iris==i, 1], label=name, s=60)
plt.title('t-SNE Visualization — Iris Dataset')
plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.legend()
plt.tight_layout()
plt.show()

print('t-SNE shows clearer cluster separation than PCA for non-linear data!')

---
## 6. Market Basket Analysis — Main Project

### What is Market Basket Analysis?
Market Basket Analysis finds **associations between products** that customers frequently buy together. It uses the **Apriori algorithm** to generate association rules.

### Key Concepts:
| Term | Definition |
|------|------------|
| **Support** | How often an itemset appears in transactions |
| **Confidence** | P(B given A was bought) |
| **Lift** | How much more likely B is bought with A vs alone (>1 = positive correlation) |

In [ ]:
# Sample Market Basket Dataset (Grocery Transactions)
transactions = [
    ['Milk', 'Bread', 'Butter'],
    ['Milk', 'Bread'],
    ['Milk', 'Eggs', 'Butter'],
    ['Bread', 'Butter', 'Eggs'],
    ['Milk', 'Bread', 'Eggs', 'Butter'],
    ['Bread', 'Eggs'],
    ['Milk', 'Eggs'],
    ['Milk', 'Bread', 'Butter', 'Cheese'],
    ['Bread', 'Butter', 'Cheese'],
    ['Milk', 'Cheese', 'Eggs'],
    ['Milk', 'Bread', 'Eggs'],
    ['Butter', 'Cheese', 'Eggs'],
    ['Milk', 'Bread', 'Butter', 'Eggs', 'Cheese'],
    ['Bread', 'Milk'],
    ['Eggs', 'Butter', 'Milk'],
    ['Cheese', 'Bread'],
    ['Milk', 'Butter'],
    ['Eggs', 'Bread', 'Cheese'],
    ['Milk', 'Eggs', 'Bread', 'Butter'],
    ['Cheese', 'Milk', 'Bread']
]

print(f'Total Transactions: {len(transactions)}')
print(f'Sample Transaction: {transactions[0]}')

In [ ]:
# Encode transactions into one-hot format
te = TransactionEncoder()
te_array = te.fit_transform(transactions)
df = pd.DataFrame(te_array, columns=te.columns_)

print('One-Hot Encoded Transaction Matrix:')
print(df.head(10))

In [ ]:
# Item Frequency Analysis
item_freq = df.sum().sort_values(ascending=False)

plt.figure(figsize=(8, 4))
item_freq.plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Item Frequency in Transactions')
plt.xlabel('Item')
plt.ylabel('Frequency')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(item_freq)

In [ ]:
# Apply Apriori Algorithm
frequent_itemsets = apriori(df, min_support=0.3, use_colnames=True)
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(lambda x: len(x))

print(f'Total Frequent Itemsets Found: {len(frequent_itemsets)}')
print()
print(frequent_itemsets.sort_values('support', ascending=False))

In [ ]:
# Generate Association Rules
rules = association_rules(frequent_itemsets, metric='lift', min_threshold=1.0)
rules = rules.sort_values('lift', ascending=False)

print(f'Total Rules Generated: {len(rules)}')
print()
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10).to_string())

In [ ]:
# Visualize: Support vs Confidence (colored by Lift)
plt.figure(figsize=(9, 6))
scatter = plt.scatter(rules['support'], rules['confidence'],
                      c=rules['lift'], cmap='RdYlGn', s=100, edgecolors='black', alpha=0.8)
plt.colorbar(scatter, label='Lift')
plt.xlabel('Support')
plt.ylabel('Confidence')
plt.title('Association Rules: Support vs Confidence (colored by Lift)')
plt.tight_layout()
plt.show()

In [ ]:
# Top 5 Rules by Lift
top_rules = rules.head(5)[['antecedents', 'consequents', 'support', 'confidence', 'lift']]
top_rules['antecedents'] = top_rules['antecedents'].apply(lambda x: ', '.join(list(x)))
top_rules['consequents'] = top_rules['consequents'].apply(lambda x: ', '.join(list(x)))

print('=== Top 5 Association Rules by Lift ===')
print(top_rules.to_string(index=False))

# Plot top rules
fig, ax = plt.subplots(figsize=(10, 4))
ax.axis('off')
table = ax.table(cellText=top_rules.round(3).values,
                 colLabels=top_rules.columns,
                 cellLoc='center', loc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.5)
plt.title('Top 5 Association Rules', fontsize=13, pad=20)
plt.tight_layout()
plt.show()

---
## 7. Summary & Key Takeaways

| Algorithm | Type | Use Case |
|-----------|------|----------|
| K-Means | Clustering | Customer segmentation |
| Hierarchical | Clustering | Gene expression, document grouping |
| PCA | Dimensionality Reduction | Feature compression, noise removal |
| t-SNE | Dimensionality Reduction | Visualization of high-dim data |
| Apriori | Association Rules | Market basket, recommendation systems |

### Market Basket Analysis Insights:
- Items with **high lift (>1)** are positively correlated — buying one increases chance of buying the other
- **Confidence** shows reliability of the rule
- **Support** shows how common the itemset is
- These rules can drive **product placement, promotions, and recommendations** in retail stores

---
*GDGOC AI/ML Fellowship — Task 12 — Week 7*